# Tuần 2 — Standard RAG Generator (Hệ 2 trong `PIPELINE.md`)

Notebook này build **Hệ 2 — Standard RAG** (theo kiến trúc ở `final/PIPELINE.md` §3): retrieve top-k cố định rồi generate thẳng, **chưa có** retrieval-decision/ISREL/ISSUP/ISUSE — 4 module đó là Notebook 3 (Self-RAG-inspired).

**Yêu cầu trước khi chạy**: đã chạy xong `01_retrieval_baseline.ipynb` — notebook này load lại `chunks.faiss`, `chunks_meta.json`, `dev_split_qids.json` từ `ARTIFACT_DIR`, không build lại từ đầu.

Output: `standard_rag_results.jsonl` trong `ARTIFACT_DIR` (mỗi dòng 1 câu hỏi: câu trả lời sinh ra + nguồn trích dẫn + câu trả lời gold) — dùng làm input cho bảng so sánh 3 hệ ở Notebook 4.

## 0. Cấu hình môi trường + API key

- Trên Colab: mount Drive **chỉ để lấy dữ liệu/artifact** (không `git clone` vào Drive — xem lý do ở `PIPELINE.md` §6.4). Mở notebook này trực tiếp từ GitHub, không clone.
- **API key Gemini**: lấy tại https://aistudio.google.com/apikey (Google AI Studio) — dùng đúng luồng này, **không** tạo key qua Google Cloud Console/Vertex AI, vì luồng đó bắt buộc bật billing cho project (đã gặp thật trước đây). Key lấy từ AI Studio dùng được ở free tier, không cần khai báo billing.
- Trên Colab: bấm biểu tượng chìa khóa 🔑 ở sidebar trái → "Add new secret" → tên `GEMINI_API_KEY`, dán key vào, bật "Notebook access". Chạy local: đặt biến môi trường `GEMINI_API_KEY`, hoặc notebook sẽ hỏi nhập trực tiếp (không hardcode key vào cell).
- *(Ghi chú lịch sử: notebook này từng chuyển tạm sang Groq sau khi gặp yêu cầu billing với Gemini — nay quay lại Gemini theo yêu cầu, dùng đúng luồng AI Studio ở trên để tránh lặp lại vấn đề billing.)*

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-genai

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/NLP-CS2308.CH203-data"  # sua neu ban dat ten khac
    DATA_DIR = os.path.join(DRIVE_DATA_ROOT, "VLQA")
    ARTIFACT_DIR = os.path.join(DRIVE_DATA_ROOT, "artifacts")

    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    DATA_DIR = os.path.join(REPO_DIR, "final", "dataset", "VLQA")
    ARTIFACT_DIR = os.path.join(REPO_DIR, "final", "artifacts")
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Nhap GEMINI_API_KEY: ")

print("DATA_DIR:", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR:", ARTIFACT_DIR, "| exists:", os.path.isdir(ARTIFACT_DIR))
print("GEMINI_API_KEY loaded:", bool(GEMINI_API_KEY))

## 1. Load artifact từ Notebook 1 (index, chunk metadata, dev split)

In [ ]:
import json
import faiss

INDEX_PATH = os.path.join(ARTIFACT_DIR, "chunks.faiss")
CHUNKS_META_PATH = os.path.join(ARTIFACT_DIR, "chunks_meta.json")
SPLIT_PATH = os.path.join(ARTIFACT_DIR, "dev_split_qids.json")

for path in [INDEX_PATH, CHUNKS_META_PATH, SPLIT_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Khong tim thay {path} - hay chay 01_retrieval_baseline.ipynb truoc")

index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_META_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
with open(SPLIT_PATH, encoding="utf-8") as f:
    dev_qids = set(json.load(f))

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_full = json.load(f)
dev_set = [ex for ex in train_full if ex["qid"] in dev_qids]

print(f"Da load index: {index.ntotal} chunk | dev set: {len(dev_set)} cau")

## 2. Embedding model + hàm `retrieve` (giữ nguyên tinh thần Notebook 1)

Notebook này chạy độc lập (không import chéo giữa các notebook) nên định nghĩa lại `retrieve`, lần này trả về nguyên `chunk` (cần `text` + `law_id` để build context cho prompt) thay vì chỉ `aid`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding device:", device)


def retrieve(query, top_chunks=50, max_k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _, idxs = index.search(q_emb, top_chunks)

    results, seen = [], set()
    for idx in idxs[0]:
        c = chunks[idx]
        if c["aid"] not in seen:
            seen.add(c["aid"])
            results.append(c)
        if len(results) >= max_k:
            break
    return results

## 3. Gemini client + dò model khả dụng

Không hardcode một model duy nhất — thử `client.models.list()` để xem tài khoản/khu vực này thực sự dùng được model nào trong `CANDIDATE_MODELS`, dùng model đầu tiên khớp; nếu việc liệt kê thất bại (một số key có thể chặn endpoint này) thì dùng nguyên `CANDIDATE_MODELS` theo thứ tự ưu tiên và để cơ chế retry/failover ở §4 tự xử lý khi model đầu không dùng được.

Thứ tự ưu tiên: `gemini-2.5-flash` (mạnh nhất, miễn phí) → `gemini-2.0-flash` → `gemini-2.5-flash-lite` → `gemini-2.0-flash-lite` (dự phòng khi các model trên cạn quota free tier trong ngày).

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

CANDIDATE_MODELS = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
]

try:
    available_models = {m.name.removeprefix("models/") for m in client.models.list()}
    chat_models = [m for m in CANDIDATE_MODELS if m in available_models] or CANDIDATE_MODELS
except Exception as e:
    print("Khong liet ke duoc model (dung nguyen CANDIDATE_MODELS):", e)
    chat_models = CANDIDATE_MODELS

GENERATOR_MODEL = chat_models[0]
print("Dang dung GENERATOR_MODEL =", GENERATOR_MODEL)

## 4. Prompt template Standard RAG

Nguyên tắc: chỉ trả lời dựa trên context được cung cấp, ép model từ chối/nói rõ "không đủ căn cứ" thay vì suy đoán khi context không đủ, và bắt buộc trích dẫn `law_id` — để Notebook 3 (ISSUP) có thể kiểm tra được câu trả lời có bám evidence hay không.

**Về rate limit của Gemini free tier**: lỗi 429 (`RESOURCE_EXHAUSTED`) có 2 dạng cần xử lý khác nhau:

- **Theo phút (RPM)**: chi tiết lỗi thường kèm `retryDelay` ngắn (vài giây đến ~60s) — cứ chờ đúng thời gian đó rồi gọi lại là qua.
- **Theo ngày (RPD)**: cạn hẳn ngân sách request/ngày của **model đó** — chờ trong phiên hiện tại không có ý nghĩa. Cách xử lý đúng là **chuyển sang model khác trong `CANDIDATE_MODELS`** — quota free tier của Gemini tính riêng theo từng model, nên `gemini-2.0-flash-lite` thường vẫn còn quota dù `gemini-2.5-flash` đã cạn. Logic này tự động ở cell bên dưới (`LONG_WAIT_THRESHOLD`).
- Muốn xem quota chính xác của tài khoản: https://aistudio.google.com/apikey → mục quota theo model (số liệu đổi theo thời gian nên không hardcode ở đây).
- Đã tắt "thinking" (`thinking_budget=0`) cho các model dòng `gemini-2.5-*` — tránh model tự chèn khối suy luận dài vào response, đồng thời tiết kiệm token/quota vì tác vụ ở đây chỉ cần trả lời trực tiếp.

In [ ]:
import re
import time
from google.genai import types
from google.genai.errors import ClientError, ServerError

PROMPT_TEMPLATE = """Ban la tro ly tu van phap luat Viet Nam. Chi tra loi dua tren cac dieu luat duoc cung cap ben duoi. Neu cac dieu luat khong du thong tin de tra loi, hay noi ro la khong du can cu thay vi suy doan. Khi tra loi, trich dan van ban luat tuong ung bang ky hieu [so] va ghi ro ma so van ban.

Cac dieu luat lien quan:
{context}

Cau hoi: {question}

Tra loi (tieng Viet, ngan gon, co trich dan):"""

LONG_WAIT_THRESHOLD = 30  # giay - vuot nguong nay coi la quota ngay cua model, khong phai per-minute
exhausted_models = set()


def strip_think(text):
    if not text:
        return text
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def build_context(retrieved):
    parts = []
    for i, c in enumerate(retrieved, start=1):
        parts.append(f"[{i}] (Van ban: {c['law_id']})\n{c['text']}")
    return "\n\n".join(parts)


def gemini_config(model, temperature):
    kwargs = {"temperature": temperature}
    if model.startswith("gemini-2.5"):
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
    return types.GenerateContentConfig(**kwargs)


def retry_delay_seconds(exc, fallback):
    match = re.search(r'"retryDelay"\s*:\s*"(\d+(?:\.\d+)?)s"', str(exc))
    return float(match.group(1)) if match else fallback


def model_queue():
    queue = [m for m in chat_models if m not in exhausted_models]
    return queue or ([GENERATOR_MODEL] if GENERATOR_MODEL not in exhausted_models else [])


def chat(prompt, temperature=0.2, max_retries=4):
    queue = model_queue()
    if not queue:
        print("Tat ca model kha dung deu dang bi khoa dai han/loi - dung lai, thu lai sau (xem aistudio.google.com/apikey).")
        return ""

    for model in queue:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=gemini_config(model, temperature),
                )
                return strip_think((response.text or "").strip())
            except ClientError as e:
                if e.code == 429:
                    wait = retry_delay_seconds(e, fallback=2 ** attempt)
                    if wait > LONG_WAIT_THRESHOLD:
                        print(f"Model {model} bi khoa dai han (retryDelay {wait:.0f}s) -> chuyen sang model tiep theo")
                        exhausted_models.add(model)
                        break
                    print(f"Rate limit model {model} (lan {attempt + 1}/{max_retries}) -> cho {wait:.1f}s")
                    time.sleep(wait)
                else:
                    print(f"Model {model} loi khong the retry (status {e.code}): {e} -> chuyen sang model tiep theo")
                    exhausted_models.add(model)
                    break
            except ServerError as e:
                wait = 2 ** attempt
                print(f"Loi server Gemini (lan {attempt + 1}/{max_retries}): {e} -> cho {wait}s")
                time.sleep(wait)
            except Exception as e:
                wait = 2 ** attempt
                print(f"Loi khac (lan {attempt + 1}/{max_retries}): {e} -> cho {wait}s")
                time.sleep(wait)

    print("Tat ca model trong queue deu that bai (khoa dai han hoac loi) o lan chay nay.")
    return ""


def generate_answer(question, retrieved):
    prompt = PROMPT_TEMPLATE.format(context=build_context(retrieved), question=question)
    return chat(prompt, temperature=0.2)

## 5. Chạy generation trên dev set (checkpoint từng câu, resume-safe)

Ghi thẳng từng kết quả vào file JSONL và `flush()` ngay — nếu Colab bị ngắt session giữa chừng, chạy lại cell này sẽ tự bỏ qua các `qid` đã có, không mất tiến độ. `MAX_QUESTIONS` đặt số nguyên để chạy thử nhanh (ví dụ 20), để `None` để chạy full dev set.

`generate_answer()` giờ tự động xoay vòng `CANDIDATE_MODELS` khi một model bị khóa dài hạn (xem `LONG_WAIT_THRESHOLD` ở cell trên), nên không cần can thiệp tay khi gặp quota giờ/ngày nữa. Nếu vẫn thấy chậm/hay bị chặn:

1. **`SLEEP_BETWEEN_CALLS`** — tăng lên (ví dụ 3-5s) để chủ động giãn request per-minute.
2. **`RETRIEVE_K`** — giảm số passage đưa vào context (ví dụ 3 thay vì 5) → giảm token/request.
3. Nếu **tất cả** model trong `CANDIDATE_MODELS` đều bị khóa dài hạn cùng lúc (in ra dòng "Tat ca model... deu bi khoa dai han"), không còn cách nào chờ được trong phiên này — hạ `MAX_QUESTIONS` và chạy tiếp vào ngày khác (`exhausted_models` chỉ tồn tại trong phiên hiện tại, phiên mới sẽ thử lại từ đầu).

In [ ]:
MAX_QUESTIONS = None
SLEEP_BETWEEN_CALLS = 2  # tang len neu van bi 429 lien tuc
RETRIEVE_K = 5  # giam xuong 3 neu muon giam token/request
RESULTS_PATH = os.path.join(ARTIFACT_DIR, "standard_rag_results.jsonl")

existing_records = []
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["generated_answer"]:
                existing_records.append(rec)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        for rec in existing_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

done_qids = {rec["qid"] for rec in existing_records}
print(f"Da co san {len(done_qids)} cau tra loi thanh cong tu lan chay truoc (da loai bo cac lan loi/rong)")

questions_to_run = dev_set if MAX_QUESTIONS is None else dev_set[:MAX_QUESTIONS]

with open(RESULTS_PATH, "a", encoding="utf-8") as f:
    for ex in questions_to_run:
        if ex["qid"] in done_qids:
            continue
        retrieved = retrieve(ex["question"], top_chunks=50, max_k=RETRIEVE_K)
        answer = generate_answer(ex["question"], retrieved)
        record = {
            "qid": ex["qid"],
            "question": ex["question"],
            "gold_answer": ex["answer"],
            "retrieved_aids": [c["aid"] for c in retrieved],
            "retrieved_law_ids": [c["law_id"] for c in retrieved],
            "generated_answer": answer,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(SLEEP_BETWEEN_CALLS)

print("Hoan tat.")

## 6. Soi vài kết quả (định tính)

So sánh câu trả lời sinh ra với câu trả lời gold — đánh giá định lượng thật (ISSUP/ISUSE) sẽ làm ở Notebook 3+4, ở đây chỉ xem nhanh chất lượng bằng mắt.

In [ ]:
import pandas as pd

results_df = pd.read_json(RESULTS_PATH, lines=True)
print(f"So cau da sinh: {len(results_df)}")

for _, row in results_df.sample(min(3, len(results_df)), random_state=0).iterrows():
    print("qid:", row["qid"])
    print("Cau hoi:", row["question"])
    print("Nguon trich dan (law_id):", row["retrieved_law_ids"])
    print("Answer sinh ra:", row["generated_answer"])
    print("Gold answer:", row["gold_answer"])
    print("-" * 80)

## 7. Bước tiếp theo

- **Notebook 3**: thêm 4 module reflection (Retrieve-decision, ISREL, ISSUP, ISUSE) lên trên cùng `retrieve()` + `generate_answer()` ở đây, tạo `self_rag_results.jsonl` cùng schema để dễ so sánh.
- **Notebook 4**: load cả `standard_rag_results.jsonl` và `self_rag_results.jsonl`, chấm ISSUP/ISUSE bằng LLM-judge, xuất `final_comparison_table.csv`.